# Atividade 6 — Desocupação e afazeres domésticos

Os percentuais mostram a composição dos desocupados por sexo no 1º trimestre de 2012 e de 2026. As horas de cuidados e afazeres domésticos são de 2022 e permanecem separadas por sexo e cor ou raça.

## 1. Leitura das duas Tabelas 5

In [1]:
import pandas as pd

t2012 = pd.read_csv(
    "Tabela5-sem_emprego_2012.csv",
    sep=";",
    decimal=",",
    encoding="utf-8-sig"
)
t2026 = pd.read_csv(
    "Tabela5-sem_emprego_2026.csv",
    sep=";",
    decimal=",",
    encoding="utf-8-sig"
)

t2012 = t2012.rename(columns={
    "Desocupados - homens (2012 T1)": "homens_2012",
    "Desocupados - mulheres (2012 T1)": "mulheres_2012"
})
t2026 = t2026.rename(columns={
    "Desocupados - homens (2026 T1)": "homens_2026",
    "Desocupados - mulheres (2026 T1)": "mulheres_2026"
})

display(t2012.head(), t2026.head())

,Sigla,Código,Estado,homens_2012,mulheres_2012
0,AC,12,Acre,45.3,54.7
1,AL,27,Alagoas,44.7,55.3
2,AM,13,Amazonas,47.5,52.5
3,AP,16,Amapá,48.1,51.9
4,BA,29,Bahia,43.8,56.2


,Sigla,Código,Estado,homens_2026,mulheres_2026
0,AC,12,Acre,53.2,46.8
1,AL,27,Alagoas,50.7,49.3
2,AM,13,Amazonas,46.8,53.2
3,AP,16,Amapá,51.1,48.9
4,BA,29,Bahia,44.7,55.3


## 2. Junção por sigla, código e estado

In [2]:
comp = t2012.merge(
    t2026,
    on=["Sigla", "Código", "Estado"],
    how="inner",
    validate="one_to_one"
)

comp

,Sigla,Código,Estado,homens_2012,mulheres_2012,homens_2026,mulheres_2026
0,AC,12,Acre,45.3,54.7,53.2,46.8
1,AL,27,Alagoas,44.7,55.3,50.7,49.3
2,AM,13,Amazonas,47.5,52.5,46.8,53.2
3,AP,16,Amapá,48.1,51.9,51.1,48.9
4,BA,29,Bahia,43.8,56.2,44.7,55.3
5,CE,23,Ceará,51.1,48.9,52.7,47.3
6,DF,53,Distrito Federal,41.0,59.0,47.9,52.1
7,ES,32,Espírito Santo,46.4,53.6,47.5,52.5
8,GO,52,Goiás,41.0,59.0,47.5,52.5
9,MA,21,Maranhão,47.2,52.8,49.1,50.9


## 3. Comparação entre 2012 e 2026

A variação está em pontos percentuais: participação em 2026 menos participação em 2012.

In [3]:
comp["variacao_homens_pp"] = (comp["homens_2026"] - comp["homens_2012"]).round(1)
comp["variacao_mulheres_pp"] = (comp["mulheres_2026"] - comp["mulheres_2012"]).round(1)

comp[["Sigla", "Estado", "variacao_homens_pp", "variacao_mulheres_pp"]].sort_values(
    "variacao_mulheres_pp", ascending=False
)

,Sigla,Estado,variacao_homens_pp,variacao_mulheres_pp
20,RO,Rondônia,-10.4,10.4
26,TO,Tocantins,-9.1,9.1
24,SE,Sergipe,-6.2,6.2
19,RN,Rio Grande do Norte,-2.8,2.8
11,MS,Mato Grosso do Sul,-1.9,1.9
12,MT,Mato Grosso,-1.3,1.3
2,AM,Amazonas,-0.7,0.7
25,SP,São Paulo,0.8,-0.8
4,BA,Bahia,0.9,-0.9
7,ES,Espírito Santo,1.1,-1.1


## 4. Leitura das horas semanais do IBGE — 2022

In [4]:
bruto = pd.read_excel(
    "Tabela 1.1.1.xls",
    sheet_name="2022",
    header=None,
    engine="xlrd"
)

horas = bruto.iloc[8:41].copy().reset_index(drop=True)
horas.columns = [
    "Estado",
    "horas_total_2022",
    "horas_branca_2022",
    "horas_preta_parda_2022",
    "horas_homens_branca_2022",
    "horas_homens_preta_parda_2022",
    "horas_mulheres_branca_2022",
    "horas_mulheres_preta_parda_2022"
]

horas["Estado"] = horas["Estado"].str.strip()
for coluna in horas.columns[1:]:
    horas[coluna] = pd.to_numeric(horas[coluna])

horas.head()

,Estado,horas_total_2022,horas_branca_2022,horas_preta_parda_2022,horas_homens_branca_2022,horas_homens_preta_parda_2022,horas_mulheres_branca_2022,horas_mulheres_preta_parda_2022
0,Brasil,16.985781,16.545930,17.333142,11.705136,11.746330,20.404106,22.037998
1,Norte,16.187037,15.925246,16.256032,11.588653,11.514646,19.295780,20.534188
2,Rondônia,17.014162,16.727733,17.101523,12.111127,12.544080,20.412887,21.061410
3,Acre,13.606903,13.806929,13.541399,10.796120,9.686983,16.130352,16.710754
4,Amazonas,16.807290,15.987545,16.995493,12.233043,12.822584,18.764969,20.716986


## 5. Cruzamento das tabelas por estado

In [5]:
horas_uf = horas[horas["Estado"].isin(comp["Estado"])].copy()

comparacao = comp.merge(
    horas_uf,
    on="Estado",
    how="left",
    validate="one_to_one"
).sort_values("Sigla").reset_index(drop=True)

print(f"Tabela final com {len(comparacao)} unidades da Federação.")
comparacao.head()

Tabela final com 27 unidades da Federação.


,Sigla,Código,Estado,homens_2012,mulheres_2012,homens_2026,mulheres_2026,variacao_homens_pp,variacao_mulheres_pp,horas_total_2022,horas_branca_2022,horas_preta_parda_2022,horas_homens_branca_2022,horas_homens_preta_parda_2022,horas_mulheres_branca_2022,horas_mulheres_preta_parda_2022
0,AC,12,Acre,45.3,54.7,53.2,46.8,7.9,-7.9,13.606903,13.806929,13.541399,10.796120,9.686983,16.130352,16.710754
1,AL,27,Alagoas,44.7,55.3,50.7,49.3,6.0,-6.0,20.217129,19.165111,20.560451,12.121839,13.357160,24.093655,25.856637
2,AM,13,Amazonas,47.5,52.5,46.8,53.2,-0.7,0.7,16.807290,15.987545,16.995493,12.233043,12.822584,18.764969,20.716986
3,AP,16,Amapá,48.1,51.9,51.1,48.9,3.0,-3.0,13.586656,14.389180,13.415091,11.116821,10.749444,16.648579,15.908487
4,BA,29,Bahia,43.8,56.2,44.7,55.3,0.9,-0.9,17.849776,17.538007,17.935566,10.551673,10.982808,22.572730,23.217119


## 6. Salvar a tabela final em CSV

In [6]:
comparacao.to_csv("comparacao_desocupacao_horas.csv", index=False, encoding="utf-8-sig")
print("Tabela comparativa salva em comparacao_desocupacao_horas.csv.")

Tabela comparativa salva em comparacao_desocupacao_horas.csv.
